# OpenDDE on Colab

Requires a GPU runtime: **Runtime > Change runtime type > T4 GPU** (or better).

This notebook: installs OpenDDE from source with `uv`, downloads model weights + inference data, runs a tiny smoke test, then runs your real prediction job.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d OpenDDE || git clone https://github.com/aurekaresearch/OpenDDE.git
%cd OpenDDE
!uv pip install --torch-backend cu126 -e '.[gpu]'
!uv pip install --group dev
!opendde doctor

In [ ]:
# General-purpose checkpoint used by default with -n opendde_v1.
%env OPENDDE_ROOT_DIR=/content/data
!mkdir -p $OPENDDE_ROOT_DIR/checkpoint

In [ ]:
# The download script needs the `zstd` CLI (not the Python `zstd` package) to
# decompress the search-database .zst files, or it silently falls back to a
# broken S3 mirror (403).
!apt-get -qq update && apt-get -qq install -y zstd
!bash /content/OpenDDE/scripts/download_opendde_data.sh
# If you don't need MSA/template/RNA-MSA features (as in the pred command below),
# add --skip-search-database to skip several GB of files you won't use:
# !bash /content/OpenDDE/scripts/download_opendde_data.sh --skip-search-database

In [ ]:
# Smoke test with the minimal example from the OpenDDE README, to confirm
# the install + data download actually work before running a real job.
tiny_json = '''[
    {
        "name": "tiny",
        "modelSeeds": [101],
        "sequences": [
            {
                "proteinChain": {
                    "sequence": "ACDEFGHIK",
                    "count": 1
                }
            }
        ]
    }
]'''
with open("tiny.json", "w") as f:
    f.write(tiny_json)

!opendde pred -i tiny.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10

In [ ]:
# Your real job. Upload/create egfr-rad51-kinase.json in this working directory
# first (it wasn't part of the original notebook, so it must exist before running).
!opendde pred -i egfr-rad51-kinase.json -o ./output -n opendde_v1 --use_msa false --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10